In [ ]:
!pip install pydicom opencv-python-headless scikit-image matplotlib numpy

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (20, 12)

In [ ]:
import cv2
import pydicom
import numpy as np
import matplotlib.pyplot as plt
from skimage.filters import threshold_otsu

SHOW_IMG_PREPROCESSING = False

#pre-processing pipeline
class MammographyPreprocessor:

    def __init__(self):
        self.stages = {}

        self.nipple_kernel = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE, (15, 15)
        )

    # -------------------------
    def load_dicom(self, path):
        ds = pydicom.dcmread(path)
        img = ds.pixel_array.astype(np.float32)

        if hasattr(ds, "RescaleSlope"):
            img = img * ds.RescaleSlope + ds.RescaleIntercept

        if ds.PhotometricInterpretation == "MONOCHROME1":
            img = np.max(img) - img

        img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX)
        img = img.astype(np.uint8)

        if SHOW_IMG_PREPROCESSING: self.stages["Raw DICOM"] = img
        return img

    # -------------------------
    def ensure_left(self, img):
        _, binary = cv2.threshold(img, 0, 255, cv2.THRESH_OTSU)
        cx = np.mean(np.where(binary > 0)[1])
        if cx > img.shape[1] / 2:
            img = cv2.flip(img, 1)

        if SHOW_IMG_PREPROCESSING: self.stages["Oriented"] = img
        return img

    # -------------------------
    def breast_mask(self, img):
        mask = img > threshold_otsu(img)
        if SHOW_IMG_PREPROCESSING: self.stages["Breast Mask"] = mask.astype(np.uint8) * 255
        return mask

    # -------------------------
    def extract_single_breast(self, img):
        mask = self.breast_mask(img)

        n, labels, stats, _ = cv2.connectedComponentsWithStats(
            mask.astype(np.uint8), 8
        )
        label = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
        component = labels == label

        ys, xs = np.where(component)
        roi = img[ys.min():ys.max(), xs.min():xs.max()]

        if SHOW_IMG_PREPROCESSING: self.stages["ROI"] = roi
        return roi

    # -------------------------
    def remove_inferior_fold_adaptive(self, roi):
        binary = roi > threshold_otsu(roi)
        row_density = binary.mean(axis=1)

        cutoff = len(row_density)
        for i in range(int(0.4 * len(row_density)), len(row_density)):
            if row_density[i] < 0.15:
                cutoff = i
                break

        roi = roi[:cutoff]
        if SHOW_IMG_PREPROCESSING: self.stages["Fold Removed"] = roi
        return roi

    # -------------------------
    def suppress_nipple(self, roi):
      binary = (roi > threshold_otsu(roi)).astype(np.uint8)

      contours, _ = cv2.findContours(
          binary,
          cv2.RETR_EXTERNAL,
          cv2.CHAIN_APPROX_SIMPLE
      )

      if not contours:
          if SHOW_IMG_PREPROCESSING: self.stages["Nipple Suppressed"] = roi
          return roi

      cnt = max(contours, key=cv2.contourArea)
      hull = cv2.convexHull(cnt)

      mask_hull = np.zeros(binary.shape, dtype=np.uint8)
      mask_cnt  = np.zeros(binary.shape, dtype=np.uint8)

      cv2.drawContours(mask_hull, [hull], -1, 1, -1)
      cv2.drawContours(mask_cnt,  [cnt],  -1, 1, -1)

      nipple = (mask_hull - mask_cnt) > 0
      nipple = cv2.dilate(
          nipple.astype(np.uint8),
          self.nipple_kernel
      )

      roi = roi.copy()
      roi[nipple > 0] = 0

      if SHOW_IMG_PREPROCESSING: self.stages["Nipple Suppressed"] = roi
      return roi


    # -------------------------
    def resize_and_color(self, img):
      """
      Step 5: Aspect-ratio–safe resizing to 720×480
      Step 6: Magma color mapping
      """

      h, w = img.shape

      # Step 5 — aspect ratio check (h / w > 3/2)
      if h / w > 1.5:
          # compute required width to satisfy h/w = 3/2
          new_w = int(np.ceil(h / 1.5))
          pad = new_w - w

          # pad ZERO columns on the RIGHT
          img = np.pad(
              img,
              pad_width=((0, 0), (0, pad)),
              mode="constant",
              constant_values=0
          )

      # final resize to (width=720, height=480)
      img = cv2.resize(
          img,
          (720, 480),
          interpolation=cv2.INTER_LINEAR
      )

      # Step 6 — convert 2D grayscale → 3D magma
      img = cv2.applyColorMap(img, cv2.COLORMAP_MAGMA)

      # OpenCV uses BGR → convert to RGB for DL frameworks / matplotlib
      img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

      if SHOW_IMG_PREPROCESSING: self.stages["Final Output"] = img

      return img



    # -------------------------
    def visualize(self):
        n = len(self.stages)
        cols = 4
        rows = int(np.ceil(n / cols))

        plt.figure()
        for i, (k, v) in enumerate(self.stages.items(), 1):
            plt.subplot(rows, cols, i)
            plt.imshow(v if v.ndim == 3 else v, cmap=None if v.ndim == 3 else "gray")
            plt.title(k)
            plt.axis("off")
        plt.show()

    # -------------------------
    def __call__(self, dicom_path):
        self.stages = {}

        img = self.load_dicom(dicom_path)
        img = self.ensure_left(img)
        roi = self.extract_single_breast(img)
        roi = self.remove_inferior_fold_adaptive(roi)
        roi = self.suppress_nipple(roi)
        roi = self.resize_and_color(roi)

        self.visualize()
        return roi


In [ ]:
import re

def map_birads_to_binary(birads):
    """
    Robust BI-RADS → binary mapping.
    Handles int, float, string formats.
    """
    if pd.isna(birads):
        return None

    # Extract number if string like "BI-RADS 2"
    if isinstance(birads, str):
        match = re.search(r"\d+", birads)
        if not match:
            return None
        birads = int(match.group())

    birads = int(birads)

    if birads in [1, 2]:
        return 0  # benign
    if birads == 5:
        return 1  # malignant

    # BI-RADS 4 or others → exclude
    return None


In [ ]:
import os
import pandas as pd
import torch
from torch.utils.data import Dataset

class VinDRMammoBinaryDataset(Dataset):
    """
    VinDr-Mammo binary classification:
    benign (0) vs malignant (1)
    """

    def __init__(
        self,
        images_root,
        csv_file,
        preprocessor,
        transform=None
    ):
        self.images_root = images_root
        self.preprocessor = preprocessor
        self.transform = transform

        df = pd.read_csv(csv_file)

        df["label"] = df["breast_birads"].apply(map_birads_to_binary)
        df = df.dropna(subset=["label"])

        df["label"] = df["label"].astype(int)

        self.samples = df[
            ["study_id", "image_id", "label"]
        ].drop_duplicates().values

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        study_id, image_id, label = self.samples[idx]

        dicom_path = os.path.join(
            self.images_root,
            str(study_id),
            f"{image_id}.dicom"
        )

        img = self.preprocessor(dicom_path)
        if img is None:
            raise RuntimeError(f"Failed preprocessing: {dicom_path}")

        img = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        label = torch.tensor(label, dtype=torch.long)

        if self.transform:
            img = self.transform(img)

        return img, label



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
pre = MammographyPreprocessor()

dataset = VinDRMammoBinaryDataset(
    images_root="/content/drive/MyDrive/vindr-mammo/images",
    csv_file="/content/drive/MyDrive/vindr-mammo/metadata/stratified_selection.csv",
    preprocessor=pre
)

img, label = dataset[0]
print(img.shape)   # torch.Size([3, 480, 720])
print(label)       # 0 or 1


<Figure size 2000x1200 with 0 Axes>

torch.Size([3, 480, 720])
tensor(1)


In [ ]:
import matplotlib.pyplot as plt

def plot_benign_malignant(dataset):
    benign_sample = None
    malignant_sample = None

    # Find one sample of each class
    for i in range(len(dataset)):
        img, label = dataset[i]
        if label.item() == 0 and benign_sample is None:
            benign_sample = img
        elif label.item() == 1 and malignant_sample is None:
            malignant_sample = img

        if benign_sample is not None and malignant_sample is not None:
            break

    if benign_sample is None or malignant_sample is None:
        raise RuntimeError("Could not find both benign and malignant samples")

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    axes[0].imshow(benign_sample.permute(1, 2, 0))
    axes[0].set_title("Benign (BI-RADS 1–2)")
    axes[0].axis("off")

    axes[1].imshow(malignant_sample.permute(1, 2, 0))
    axes[1].set_title("Malignant (BI-RADS 5)")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
plot_benign_malignant(dataset)


<Figure size 2000x1200 with 0 Axes>

<Figure size 2000x1200 with 0 Axes>

<Figure size 2000x1200 with 0 Axes>

KeyboardInterrupt: 

In [ ]:
# def plot_grid(dataset, label_value, title, n=4):
#     imgs = []
#     for i in range(len(dataset)):
#         img, label = dataset[i]
#         if label.item() == label_value:
#             imgs.append(img)
#         if len(imgs) == n:
#             break

#     fig, axes = plt.subplots(1, n, figsize=(4*n, 4))
#     for ax, img in zip(axes, imgs):
#         ax.imshow(img.permute(1,2,0))
#         ax.axis("off")

#     fig.suptitle(title, fontsize=14)
#     plt.show()

# plot_grid(dataset, 0, "Benign Samples")
# plot_grid(dataset, 1, "Malignant Samples")

In [ ]:
import matplotlib.pyplot as plt

def plot_label_distribution(dataset):
    benign = 0
    malignant = 0

    for i in range(len(dataset)):
        _, label = dataset[i]
        if label.item() == 0:
            benign += 1
        elif label.item() == 1:
            malignant += 1

    labels = ["Benign", "Malignant"]
    counts = [benign, malignant]

    plt.figure(figsize=(5, 4))
    plt.bar(labels, counts)
    plt.ylabel("Number of Images")
    plt.title("Benign vs Malignant Distribution (VinDr-Mammo)")

    for i, v in enumerate(counts):
        plt.text(i, v, str(v), ha="center", va="bottom")

    plt.tight_layout()
    plt.show()

plot_label_distribution(dataset)


<Figure size 2000x1200 with 0 Axes>

<Figure size 2000x1200 with 0 Axes>

<Figure size 2000x1200 with 0 Axes>

<Figure size 2000x1200 with 0 Axes>

<Figure size 2000x1200 with 0 Axes>

<Figure size 2000x1200 with 0 Axes>

<Figure size 2000x1200 with 0 Axes>

KeyboardInterrupt: 

In [ ]:
import torch
from collections import Counter

labels = [dataset[i][1].item() for i in range(len(dataset))]
counts = Counter(labels)

print("Class counts:", counts)

# Binary case
n_benign = counts[0]
n_malignant = counts[1]

pos_weight = torch.tensor([n_benign / n_malignant])
print("pos_weight:", pos_weight.item())


<Figure size 2000x1200 with 0 Axes>

<Figure size 2000x1200 with 0 Axes>

<Figure size 2000x1200 with 0 Axes>

<Figure size 2000x1200 with 0 Axes>

<Figure size 2000x1200 with 0 Axes>

...

<Figure size 2000x1200 with 0 Axes>

<Figure size 2000x1200 with 0 Axes>

<Figure size 2000x1200 with 0 Axes>

<Figure size 2000x1200 with 0 Axes>

Class counts: Counter({0: 410, 1: 226})
pos_weight: 1.8141592741012573


In [ ]:
import torch
import numpy as np

def collect_predictions(model, dataloader, device):
    model.eval()

    all_probs = []
    all_labels = []

    with torch.no_grad():
        for imgs, labels in dataloader:
            imgs = imgs.to(device)
            labels = labels.to(device)

            logits = model(imgs).squeeze(1)  # [B]
            probs = torch.sigmoid(logits)    # convert to probabilities

            all_probs.append(probs.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    all_probs = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)

    return all_labels, all_probs


In [ ]:
import torch
import torch.nn as nn
from torchvision import models

def build_resnet152(num_classes=1, pretrained=True):
    model = models.resnet152(pretrained=pretrained)

    # Replace final FC layer
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)

    return model


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = build_resnet152(pretrained=True)
model = model.to(device)


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet152_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet152_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet152-394f9c45.pth" to /root/.cache/torch/hub/checkpoints/resnet152-394f9c45.pth


100%|██████████| 230M/230M [00:01<00:00, 181MB/s]


In [ ]:
criterion = torch.nn.BCEWithLogitsLoss(
    pos_weight=pos_weight.to(device)
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)


In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0

    for imgs, labels in loader:
        imgs = imgs.to(device)
        labels = labels.float().to(device)

        optimizer.zero_grad()

        logits = model(imgs).squeeze(1)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)

    return running_loss / len(loader.dataset)


In [ ]:
from sklearn.metrics import roc_auc_score
import numpy as np

def validate(model, loader, device):
    model.eval()
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            labels = labels.to(device)

            logits = model(imgs).squeeze(1)
            probs = torch.sigmoid(logits)

            all_probs.append(probs.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    all_probs = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)

    auc = roc_auc_score(all_labels, all_probs)
    return auc


In [ ]:
num_epochs = 60

from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
import numpy as np

# Extract labels
labels = [dataset[i][1].item() for i in range(len(dataset))]

indices = np.arange(len(dataset))

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.2,
    stratify=labels,
    random_state=42
)

train_dataset = Subset(dataset, train_idx)
val_dataset = Subset(dataset, val_idx)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,      # reduce if OOM
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)





<Figure size 2000x1200 with 0 Axes>

<Figure size 2000x1200 with 0 Axes>

<Figure size 2000x1200 with 0 Axes>

<Figure size 2000x1200 with 0 Axes>

<Figure size 2000x1200 with 0 Axes>

...

<Figure size 2000x1200 with 0 Axes>

<Figure size 2000x1200 with 0 Axes>

<Figure size 2000x1200 with 0 Axes>

<Figure size 2000x1200 with 0 Axes>

<Figure size 2000x1200 with 0 Axes>

In [ ]:
imgs, labels = next(iter(train_loader))
print("Batch image shape:", imgs.shape)
print("Batch labels:", labels)


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7c044cf21f80>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1654, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1618, in _shutdown_workers
    w.join(timeout=_utils.MP_STATUS_CHECK_INTERVAL)
  File "/usr/lib/python3.12/multiprocessing/process.py", line 149, in join
    res = self._popen.wait(timeout)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/popen_fork.py", line 40, in wait
    if not wait([self.sentinel], timeout):
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 1136, in wait
    ready = selector.select(timeout)
            ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/selectors.py", line 415, in select
    fd_event_list = self._selector.poll(timeout)
    

Batch image shape: torch.Size([4, 3, 480, 720])
Batch labels: tensor([0, 1, 1, 0])


In [ ]:
from collections import Counter

def count_labels(loader):
    counter = Counter()
    for _, labels in loader:
        counter.update(labels.tolist())
    return counter

print("Train distribution:", count_labels(train_loader))
print("Val distribution:", count_labels(val_loader))

Train distribution: Counter({0: 327, 1: 181})
Val distribution: Counter({0: 83, 1: 45})


In [ ]:
num_epochs = 10

for epoch in range(num_epochs):
    train_loss = train_one_epoch(
        model, train_loader, optimizer, criterion, device
    )

    val_auc = validate(model, val_loader, device)

    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val AUC: {val_auc:.4f}"
    )


Epoch 1/10 | Train Loss: 0.7168 | Val AUC: 0.9135
Epoch 2/10 | Train Loss: 0.6687 | Val AUC: 0.9277
Epoch 3/10 | Train Loss: 0.5939 | Val AUC: 0.9189
Epoch 4/10 | Train Loss: 0.5008 | Val AUC: 0.8541
Epoch 5/10 | Train Loss: 0.4646 | Val AUC: 0.9606
Epoch 6/10 | Train Loss: 0.4355 | Val AUC: 0.9336
Epoch 7/10 | Train Loss: 0.3618 | Val AUC: 0.8972
Epoch 8/10 | Train Loss: 0.2842 | Val AUC: 0.9448
Epoch 9/10 | Train Loss: 0.3813 | Val AUC: 0.9523
Epoch 10/10 | Train Loss: 0.2347 | Val AUC: 0.9548


In [ ]:
from sklearn.metrics import roc_auc_score

y_true, y_score = collect_predictions(model, val_loader, device)

auc = roc_auc_score(y_true, y_score)
print(f"ROC–AUC: {auc:.4f}")


ROC–AUC: 0.9548


In [ ]:
import numpy as np
from sklearn.metrics import precision_score, recall_score

threshold = 0.5
y_pred = (y_score >= threshold).astype(int)

precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)

print(f"Precision @ 0.5: {precision:.3f}")
print(f"Recall @ 0.5: {recall:.3f}")


Precision @ 0.5: 0.925
Recall @ 0.5: 0.822


In [ ]:
from sklearn.metrics import roc_curve
import numpy as np

def threshold_at_specificity(y_true, y_score, target_specificity=0.90):
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    specificity = 1 - fpr

    valid = np.where(specificity >= target_specificity)[0]
    if len(valid) == 0:
        return None

    best = valid[np.argmax(tpr[valid])]
    return thresholds[best]


In [ ]:
thr_90 = threshold_at_specificity(y_true, y_score, 0.90)
print("Threshold @ 90% specificity:", thr_90)


Threshold @ 90% specificity: 0.44078478


In [ ]:
y_pred_90 = (y_score >= thr_90).astype(int)

precision_90 = precision_score(y_true, y_pred_90)
recall_90 = recall_score(y_true, y_pred_90)

print(f"Precision @ 90% specificity: {precision_90:.3f}")
print(f"Recall @ 90% specificity: {recall_90:.3f}")


Precision @ 90% specificity: 0.929
Recall @ 90% specificity: 0.867


In [ ]:
from sklearn.metrics import confusion_matrix, roc_auc_score, precision_score, recall_score
import numpy as np

# Collect predictions on the INbreast dataset
inbreast_y_true, inbreast_y_score = collect_predictions(model, inbreast_loader, device)

# Evaluate ROC-AUC
inbreast_auc = roc_auc_score(inbreast_y_true, inbreast_y_score)
print(f"INbreast ROC-AUC: {inbreast_auc:.4f}")

# Evaluate Precision and Recall at a default threshold (e.g., 0.5)
inbreast_threshold = 0.5
inbreast_y_pred = (inbreast_y_score >= inbreast_threshold).astype(int)

inbreast_precision = precision_score(inbreast_y_true, inbreast_y_pred)
inbreast_recall = recall_score(inbreast_y_true, inbreast_y_pred)

print(f"INbreast Precision @ {inbreast_threshold}: {inbreast_precision:.3f}")
print(f"INbreast Recall @ {inbreast_threshold}: {inbreast_recall:.3f}")

# Evaluate Precision and Recall at 90% specificity (using the function defined earlier)
inbreast_thr_90 = threshold_at_specificity(inbreast_y_true, inbreast_y_score, target_specificity=0.90)
if inbreast_thr_90 is not None:
    print(f"INbreast Threshold @ 90% specificity: {inbreast_thr_90:.4f}")
    inbreast_y_pred_90 = (inbreast_y_score >= inbreast_thr_90).astype(int)

    inbreast_precision_90 = precision_score(inbreast_y_true, inbreast_y_pred_90)
    inbreast_recall_90 = recall_score(inbreast_y_true, inbreast_y_pred_90)

    print(f"INbreast Precision @ 90% specificity: {inbreast_precision_90:.3f}")
    print(f"INbreast Recall @ 90% specificity: {inbreast_recall_90:.3f}")

    # Display confusion matrix at 90% specificity
    inbreast_cm = confusion_matrix(inbreast_y_true, inbreast_y_pred_90)
    print("\nINbreast Confusion Matrix (at 90% specificity):")
    print(inbreast_cm)
else:
    print("Could not find a threshold for INbreast at 90% specificity.")


[[80  3]
 [ 6 39]]


In [ ]:
import joblib
joblib.dump(model, "/content/drive/MyDrive/vindr-mammo/model-trained.pkl")


['/content/drive/MyDrive/vindr-mammo/model-trained.pkl']

In [ ]:
from pathlib import Path
import os # Import os module for os.path.exists

class INbreastDataset(Dataset):
    def __init__(
        self,
        dicom_dir,
        csv_file,
        preprocessor,
        return_id=False
    ):
        """
        Zero-shot compatible INbreast dataset

        Args:
            dicom_dir (str): path to AllDICOMs/
            csv_file (str): INbreast annotation CSV
            preprocessor (callable): MammographyPreprocessor
            return_id (bool): whether to return image id
        """
        self.dicom_dir = Path(dicom_dir)
        self.pre = preprocessor
        self.return_id = return_id

        df = pd.read_csv(csv_file)

        # ---- REQUIRED columns (adjust names if needed) ----
        # Common INbreast fields:
        # - File Name
        # - Biopsy result / Pathology
        # -----------------------------------------------

        samples = []

        for _, row in df.iterrows():
            file_id = str(row["File Name"]).strip()

            birads_raw = str(row["Bi-Rads"]).lower().strip()

            # ---- Extract BI-RADS number robustly ----
            match = re.search(r"\b([1-6])\b", birads_raw)
            if match is None:
                continue

            birads_num = int(match.group(1))

            # ---- Exclude all BI-RADS 4 variants ----
            if birads_num == 4:
                continue

            # ---- Label mapping ----
            if birads_num in [2, 3]:
                label = 0  # benign
            elif birads_num in [5, 6]:
                label = 1  # malignant
            else:
                continue


            path = self.find_inbreast_dicom(file_id)

            # Check if path was found and exists before appending
            if path is not None and os.path.exists(path):
                samples.append((path, label, file_id))
            else:
                print(f"Warning: Could not find or access DICOM file for ID: {file_id} at expected path: {path}. Skipping sample.")
                continue

        self.samples = samples
        print(f"[INbreast] Loaded {len(self.samples)} samples")



    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        dicom_path, label, image_id = self.samples[idx]

        img = self.pre(dicom_path)  # (H, W, 3), numpy
        img = torch.from_numpy(img).permute(2, 0, 1).float()

        label = torch.tensor(label).long()
        return img, label


    def find_inbreast_dicom(self, file_id):
        file_id_str = str(file_id).strip()
        try:
            # Convert "22678646.0" to "22678646" for matching filenames
            file_id_int_part = str(int(float(file_id_str)))
        except ValueError:
            # If it's not a number that can be converted to float then int, skip
            return None

        if not self.dicom_dir.exists():
            return None

        for f in self.dicom_dir.iterdir():
            # Check for patterns like "123456_" or just "123456"
            if f.name.startswith(f"{file_id_int_part}_") or f.stem == file_id_int_part:
                return str(f)
        return None


In [ ]:
inbreast_dataset = INbreastDataset(
    dicom_dir="/content/drive/MyDrive/INbreast/AllDICOMs",
    csv_file="/content/drive/MyDrive/INbreast/INbreast.csv",
    preprocessor=pre
)

[INbreast] Loaded 300 samples


In [ ]:
# import os

# # List contents of the DICOM directory for inspection
# DICOM_DIR = "/content/drive/MyDrive/INbreast/AllDICOMs"
# if os.path.exists(DICOM_DIR):
#     print(f"Contents of {DICOM_DIR}:")
#     for i, item in enumerate(os.listdir(DICOM_DIR)):
#         print(item)
#         if i >= 10: # Print only first 10 items to avoid excessive output
#             print("...")
#             break
# else:
#     print(f"Directory not found: {DICOM_DIR}")

# # Also load and display the head of the CSV file to see 'File Name' format
# CSV_FILE = "/content/drive/MyDrive/INbreast/INbreast.csv"
# if os.path.exists(CSV_FILE):
#     df_inbreast = pd.read_csv(CSV_FILE)
#     print(f"\nHead of {CSV_FILE}:")
#     print(df_inbreast[['File Name', 'Bi-Rads']].head())
# else:
#     print(f"CSV file not found: {CSV_FILE}")

# inbreast_dataset = INbreastDataset(
#     dicom_dir=DICOM_DIR,
#     csv_file=CSV_FILE,
#     preprocessor=pre
# )

# from torch.utils.data import DataLoader

# inbreast_loader = DataLoader(
#     inbreast_dataset,
#     batch_size=4,
#     shuffle=False, # No need to shuffle for evaluation
#     num_workers=2,
#     pin_memory=True
# )

Contents of /content/drive/MyDrive/INbreast/AllDICOMs:
20586908_6c613a14b80a8591_MG_R_CC_ANON.dcm
20586934_6c613a14b80a8591_MG_L_CC_ANON.dcm
20586960_6c613a14b80a8591_MG_R_ML_ANON.dcm
20586986_6c613a14b80a8591_MG_L_ML_ANON.dcm
20587054_b6a4f750c6df4f90_MG_R_CC_ANON.dcm
20587080_b6a4f750c6df4f90_MG_R_ML_ANON.dcm
20587148_fd746d25eb40b3dc_MG_R_CC_ANON.dcm
20587174_fd746d25eb40b3dc_MG_L_CC_ANON.dcm
20587200_fd746d25eb40b3dc_MG_R_ML_ANON.dcm
20587226_fd746d25eb40b3dc_MG_L_ML_ANON.dcm
20587294_e634830794f5c1bd_MG_R_CC_ANON.dcm
...

Head of /content/drive/MyDrive/INbreast/INbreast.csv:
    File Name Bi-Rads
0  22678622.0       1
1  22678646.0       3
2  22678670.0       1
3  22678694.0       3
4  22614074.0       5
[INbreast] Loaded 300 samples


In [ ]:
from torch.utils.data import DataLoader

inbreast_loader = DataLoader(
    inbreast_dataset,
    batch_size=4,
    shuffle=False,      # ❗ NEVER shuffle evaluation
    num_workers=2,
    pin_memory=True
)


In [ ]:
img, label = inbreast_dataset[0]

print(img.shape)     # torch.Size([3, 480, 720])
print(label)         # 0 or 1
print(img.min(), img.max())


<Figure size 2000x1200 with 0 Axes>

torch.Size([3, 480, 720])
tensor(0)
tensor(0.) tensor(254.)


In [ ]:
import torch
import numpy as np

all_probs = []
all_labels = []

with torch.no_grad():
    for imgs, labels in inbreast_loader:
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = model(imgs).squeeze(1)   # [B]
        probs = torch.sigmoid(logits)     # cancer probability

        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.cpu().numpy())


In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score

y_true = np.concatenate(all_labels)
y_score = np.concatenate(all_probs)

print("INbreast ROC–AUC:", roc_auc_score(y_true, y_score))

NameError: name 'all_labels' is not defined

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

fpr_v, tpr_v, _ = roc_curve(y_vindr, y_vindr_score)
fpr_i, tpr_i, _ = roc_curve(y_true, y_score)

plt.plot(fpr_v, tpr_v, label="VinDr")
plt.plot(fpr_i, tpr_i, label="INbreast (zero-shot)")
plt.plot([0,1],[0,1],'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.title("Cross-dataset ROC")
plt.show()


NameError: name 'y_vindr' is not defined